# Building small groups from structure: automorphisms and cyclic extensions

Instead of searching all $n^{n^2}$ tables, build the groups of order $n$ from the groups of smaller orders.

**The idea.** Every solvable group $G$ has a normal subgroup $N$ of prime index $p$ (pull back a quotient of prime order from the abelian group $G/G'$). Pick any $t \in G \setminus N$. Then

* conjugation by $t$ is an automorphism $\alpha$ of $N$,
* $c = t^p$ lies in $N$, and
* $\alpha(c) = c$ and $\alpha^p = $ conjugation by $c$ (because $t^p$ commutes with $t$, and $t^p$ acts on $N$ as $c$ does).

Conversely, **every** pair $(\alpha, c)$ satisfying those two conditions defines a group of order $p\,|N|$ with elements $x\,t^i$ ($x \in N$, $0 \le i < p$) and multiplication

$$(x\,t^i)(y\,t^j) = x\,\alpha^i(y)\,t^{i+j}, \qquad t^p = c .$$

When $c = e$ this is the semidirect product $N \rtimes \mathbb{Z}_p$ that `Group.semidirect_product` builds; $c \ne e$ gives the *non-split* cases (e.g. $Q_8$, $\mathbb{Z}_8$, the dicyclic groups). The same construction works for any $m \ge 2$ in place of $p$ (then $G/N \cong \mathbb{Z}_m$).

So the plan is: **(1)** compute $\mathrm{Aut}(N)$, **(2)** enumerate the valid $(\alpha, c)$ and build each group, **(3)** recurse over the primes dividing $n$, removing duplicates with `fast_isomorphic`. Every group of order below 60 is solvable, so this should find *all* groups of every order up to 59, including the three order-32 groups that none of the other constructions reach.

This notebook only uses the public `finalg` API plus two of its private helpers (`_smallest_generating_set` and `_element_invariants`, the ones `fast_isomorphic` uses). It does not change the repo.

In [1]:
import itertools as it
from collections import defaultdict

import numpy as np
from sympy import factorint

from finalg import make_finite_algebra, identify_small_group
from finalg.group import Group
import time

from finalg import generate_cyclic_group
from finalg.group_catalog import small_groups

Finite Algebras:
  To see a listing of built-in example algebras, run finalg.examples.about()
  To retrieve an example algebra, use finalg.examples[INDEX]



## 1. Working on the integer table

Everything below works on the integer Cayley table (`G.table.table`, a NumPy array) and only builds `Group` objects when needed. An automorphism is stored as a tuple `f` where `f[i]` is the index, in `G.elements`, of the image of element `i`.

In [2]:
def int_view(G):
    """(T, e): the Cayley table of G as a NumPy int array, and the index of its identity."""
    T = np.asarray(G.table.table, dtype=np.int64)
    e = list(G.elements).index(G.identity)
    return T, e


def inverse_array(T, e):
    inv = np.empty(len(T), dtype=np.int64)
    rows, cols = np.where(T == e)
    inv[rows] = cols
    return inv


def element_orders(T, e):
    n = len(T)
    x = np.arange(n)
    pw = x.copy()
    order = np.ones(n, dtype=np.int64)
    todo = pw != e
    while todo.any():
        pw[todo] = T[pw[todo], x[todo]]
        order[todo] += 1
        todo = pw != e
    return order


def cheap_invariant(T, e):
    """An isomorphism invariant that is cheap to compute: the multiset, over the elements x, of
    (order of x, size of the centralizer of x, order of x^2, number of y with y^2 = x)."""
    n = len(T)
    order = element_orders(T, e)
    cent = (T == T.T).sum(axis=1)
    sq = T[np.arange(n), np.arange(n)]                   # x^2
    roots = np.bincount(sq, minlength=n)                 # number of square roots of each element
    return tuple(sorted(zip(order.tolist(), cent.tolist(), order[sq].tolist(), roots.tolist())))

## 2. `automorphisms`, `inner_automorphisms`, `automorphism_group`

An automorphism is determined by where it sends a generating set. So, as in `fast_isomorphic`, only the images of a small generating set are searched, restricted to elements with the same invariants (order, row/column profile, centralizer size). Each choice of images is extended by `f(a*g) = f(a)*f(g)` for every generator `g`; a conflict, or a failure to be a bijection, rejects it. Because every edge `a -> a*g` is checked, an assignment that survives is automatically a homomorphism.

In [3]:
def _extend(Tl, e, n, gens, imgs):
    """Extend {gens[i] -> imgs[i]} to f with f(a*g) = f(a)*f(g) for every a and every generator g.
    Returns f as a list, or None if the assignment is inconsistent or not a bijection."""
    f = [-1] * n
    f[e] = e
    for g, im in zip(gens, imgs):
        f[g] = im
    pairs = list(zip(gens, imgs))
    queue, seen = [e], {e}
    for a in queue:
        row, fa = Tl[a], Tl[f[a]]
        for g, im in pairs:
            p, v = row[g], fa[im]
            if f[p] == -1:
                f[p] = v
            elif f[p] != v:
                return None
            if p not in seen:
                seen.add(p)
                queue.append(p)
    if len(seen) != n or len(set(f)) != n:
        return None
    return f


def automorphisms(G):
    """All automorphisms of the Group G, as a list of tuples f, where f[i] is the index (in
    G.elements) of the image of the element with index i."""
    T, e = int_view(G)
    n = len(T)
    if n == 1:
        return [(0,)]
    names = list(G.elements)
    gens = [names.index(g) for g in G._smallest_generating_set()]   # an automorphism is determined by
    invariants = G._element_invariants()                            # its values on a generating set
    by_inv = defaultdict(list)
    for i, nm in enumerate(names):
        by_inv[invariants[nm]].append(i)
    candidates = [by_inv[invariants[names[g]]] for g in gens]       # only elements that look alike
    Tl = T.tolist()
    result = []
    for imgs in it.product(*candidates):
        if len(set(imgs)) < len(imgs):
            continue
        f = _extend(Tl, e, n, gens, imgs)
        if f is not None:
            result.append(tuple(f))
    return result


def inner_automorphism_array(T, e):
    """Row x is the inner automorphism a -> x*a*x^-1, for every element x."""
    n = len(T)
    inv, A = inverse_array(T, e), np.arange(n)
    return np.array([T[T[x, A], inv[x]] for x in range(n)], dtype=np.int64)


def inner_automorphisms(G):
    T, e = int_view(G)
    return sorted({tuple(r) for r in inner_automorphism_array(T, e).tolist()})


def automorphism_group(G, max_order=2000, return_maps=False):
    """The automorphism group of G, as a finalg Group whose elements are named f0 (the identity), f1, ...
    If return_maps is True, also return the list of automorphisms (tuples of indices) that the names stand for."""
    auts = automorphisms(G)
    k, n = len(auts), G.order
    if k > max_order:
        raise ValueError(f"|Aut({G.name})| = {k} > max_order = {max_order}")
    ident = tuple(range(n))
    auts.sort(key=lambda f: f != ident)                  # identity first
    A = np.array(auts, dtype=np.int64)
    index = {A[i].tobytes(): i for i in range(k)}
    table = np.empty((k, k), dtype=np.int64)
    for i in range(k):                                   # f_i o f_j has index table[i][j]
        table[i] = [index[row.tobytes()] for row in A[i][A]]
    Aut = make_finite_algebra(f"Aut({G.name})", f"Automorphism group of {G.name}",
                              [f"f{i}" for i in range(k)], table.tolist())
    return (Aut, auts) if return_maps else Aut

In [4]:
def cat(order, name):
    """A group from finalg's catalog of small groups, by name."""
    return next(g for g in small_groups(order) if g.name == name)

groups = [cat(5, "Z5"), cat(8, "Z8"), cat(4, "V4"), cat(8, "Z4_x_Z2"), cat(6, "D3"),
          cat(8, "D4"), cat(8, "Q8"), cat(8, "Z2_x_Z2_x_Z2")]

print(f"{'G':<14}{'|G|':>4}{'|Aut|':>7}{'|Inn|':>7}{'|Out|':>7}   Aut(G) is isomorphic to")
for G in groups:
    Aut = automorphism_group(G)
    inn = len(inner_automorphisms(G))
    s = identify_small_group(Aut)
    print(f"{G.name:<14}{G.order:>4}{Aut.order:>7}{inn:>7}{Aut.order // inn:>7}   {s.name if s else f'(order {Aut.order})'}")

G              |G|  |Aut|  |Inn|  |Out|   Aut(G) is isomorphic to
Z5               5      4      1      4   Z4
Z8               8      4      1      4   V4
V4               4      6      1      6   D3
Z4_x_Z2          8      8      1      8   D4
D3               6      6      6      1   D3
D4               8      8      4      2   D4
Q8               8     24      4      6   S4
Z2_x_Z2_x_Z2     8    168      1    168   (order 168)


In [5]:
# Larger: Aut(Z2^4) = GL(4,2) has 20160 elements
E4 = cat(16, "Z2_x_Z2_x_Z2_x_Z2")
t0 = time.perf_counter()
print(f"|Aut(Z2^4)| = {len(automorphisms(E4))}   ({time.perf_counter() - t0:.1f} s)")

|Aut(Z2^4)| = 20160   (0.2 s)


## 3. `cyclic_extensions`

`valid_extension_data` filters the pairs $(\alpha, c)$ by the two conditions. Two pairs related by an automorphism $\beta$ of $N$, $(\alpha, c) \mapsto (\beta\alpha\beta^{-1}, \beta(c))$, give isomorphic groups (via $x\,t^i \mapsto \beta(x)\,t^i$), so `reduce_by_conjugacy` keeps one pair per orbit. That is what keeps the number of groups to build small. Different orbits can still give isomorphic groups, so the survivors are compared with `fast_isomorphic`, after bucketing them by a cheap invariant.

In [6]:
def valid_extension_data(N, m, auts=None):
    """All pairs (alpha, c), alpha an automorphism of N (tuple) and c an element index of N, with
    alpha(c) = c and alpha^m = conjugation by c: exactly the data that define a group G containing N
    as a normal subgroup, with G/N cyclic of order m, namely G = <N, t> with t^m = c, t x t^-1 = alpha(x)."""
    T, e = int_view(N)
    n = len(T)
    auts = automorphisms(N) if auts is None else auts
    inn = inner_automorphism_array(T, e)
    c_with_inner = defaultdict(list)                     # conjugation-by-c map -> all c that give it
    for c in range(n):
        c_with_inner[inn[c].tobytes()].append(c)
    data = []
    for f in auts:
        F = np.array(f, dtype=np.int64)
        fm = np.arange(n)
        for _ in range(m):
            fm = F[fm]                                   # alpha^m
        for c in c_with_inner.get(fm.tobytes(), ()):
            if f[c] == c:
                data.append((f, c))
    return data


def reduce_by_conjugacy(data, auts):
    """Keep one (alpha, c) from each orbit of Aut(N) acting by (alpha, c) -> (b alpha b^-1, b(c)).
    Pairs in the same orbit give isomorphic groups (n t^i -> b(n) t^i)."""
    A = np.array(auts, dtype=np.int64)                   # row b is the automorphism b
    Ainv = np.argsort(A, axis=1)                         # row b is b^-1
    rows = np.arange(len(auts))[:, None]
    seen, reps = set(), []
    for f, c in data:
        key = (np.array(f, dtype=np.int64).tobytes(), c)
        if key in seen:
            continue
        reps.append((f, c))
        F = np.array(f, dtype=np.int64)
        conj = A[rows, F[Ainv]]                          # row b is b o alpha o b^-1
        cs = A[:, c]                                     # b(c)
        seen.update(zip((r.tobytes() for r in conj), cs.tolist()))
    return reps

In [7]:
def build_cyclic_extension(N, m, alpha, c, name=None, description=None):
    """The group G = <N, t>, t^m = c, t x t^-1 = alpha(x), as a finalg Group. Its elements are
    the pairs "x:i", standing for x t^i (x in N, 0 <= i < m), in the order of itertools.product, so
    (x t^i)(y t^j) = x alpha^i(y) t^(i+j), with t^m replaced by c."""
    T, e = int_view(N)
    n = len(T)
    A = np.arange(n)
    F = np.array(alpha, dtype=np.int64)
    pw = [A]
    for _ in range(1, m):
        pw.append(F[pw[-1]])                             # alpha^i
    table = np.empty((n * m, n * m), dtype=np.int64)
    for i in range(m):
        X = T[A[:, None], pw[i][None, :]]                # x * alpha^i(y)
        Xc = T[X, c]                                     # ... * c, for when i + j >= m
        for j in range(m):
            table[i::m, j::m] = (Xc if i + j >= m else X) * m + (i + j) % m
    delim = N.direct_product_delimiter()
    elems = [f"{x}{delim}{i}" for x in N.elements for i in range(m)]
    if name is None:
        name = f"{N.name}_ext{m}"
    if description is None:
        moved = {N.elements[a]: N.elements[b] for a, b in enumerate(alpha) if a != b}
        description = (f"Cyclic extension of {N.name} by Z{m}: t^{m} = {N.elements[c]}, "
                       f"t x t^-1 = alpha(x), alpha = {moved if moved else 'identity'}")
    G = make_finite_algebra(name, description, elems, table.tolist())
    if not isinstance(G, Group):
        raise ValueError("Internal error: the extension data did not define a group.")
    return G


def cyclic_extensions(N, m, return_data=False):
    """All groups G, up to isomorphism, that contain N as a normal subgroup with G/N cyclic of order m.
    If return_data is True, return a list of (G, alpha, c) instead, where alpha is a tuple of indices
    and c an index into N.elements (c = identity means that the extension splits)."""
    auts = automorphisms(N)
    reps = reduce_by_conjugacy(valid_extension_data(N, m, auts), auts)
    found, buckets = [], defaultdict(list)
    for alpha, c in reps:
        G = build_cyclic_extension(N, m, alpha, c, name=f"{N.name}_ext{m}_{len(found) + 1}")
        key = cheap_invariant(*int_view(G))
        if any(G.fast_isomorphic(H) for H, _, _ in buckets[key]):
            continue
        buckets[key].append((G, alpha, c))
        found.append((G, alpha, c))
    return found if return_data else [G for G, _, _ in found]

### Examples

All groups of order 8 that contain a cyclic $\mathbb{Z}_4$ as a normal subgroup of index 2. Note the two non-split cases, $c \ne 0$: $\mathbb{Z}_8$ (where $t^2$ generates $\mathbb{Z}_4$) and $Q_8$ (where $t^2$ is the element of order 2 and $t$ inverts $\mathbb{Z}_4$).

In [8]:
Z4 = cat(4, "Z4")
print("Extensions of Z4 by Z2   (c = t^2; c = 0 means that the extension splits)")
for G, alpha, c in cyclic_extensions(Z4, 2, return_data=True):
    moved = {Z4.elements[a]: Z4.elements[b] for a, b in enumerate(alpha) if a != b}
    print(f"  {identify_small_group(G).name:<9} c = {Z4.elements[c]}   alpha moves: {moved or 'nothing'}")

Extensions of Z4 by Z2   (c = t^2; c = 0 means that the extension splits)
  Z4_x_Z2   c = 0   alpha moves: nothing
  Z8        c = 1   alpha moves: nothing
  D4        c = 0   alpha moves: {'1': '3', '3': '1'}
  Q8        c = 2   alpha moves: {'1': '3', '3': '1'}


In [9]:
V4, Z3 = cat(4, "V4"), cat(3, "Z3")
for N, m in [(V4, 2), (Z3, 2), (Z3, 4), (cat(5, "Z5"), 4)]:
    names = [identify_small_group(G).name for G in cyclic_extensions(N, m)]
    print(f"Extensions of {N.name:<3} by Z{m}: {names}")

Extensions of V4  by Z2: ['Z2_x_Z2_x_Z2', 'Z4_x_Z2', 'D4']
Extensions of Z3  by Z2: ['Z6', 'D3']
Extensions of Z3  by Z4: ['Z12', 'Dic3']
Extensions of Z5  by Z4: ['Z20', 'F20', 'Dic5']


Every group built this way really does contain a copy of $N$ (the elements `x:0`) as a normal subgroup with a cyclic quotient of order $m$. Here that is checked independently with finalg's own `is_normal` and quotient group, for all of the examples above:

In [10]:
def check_extension(G, N, m):
    """Check that G has a normal subgroup isomorphic to N (the elements 'x:0') with a cyclic quotient of order m."""
    delim = N.direct_product_delimiter()
    sub = G.subalgebra_from_elements([f"{x}{delim}0" for x in N.elements], name="N_in_G", desc="copy of N")
    assert G.is_normal(sub), "N is not normal in G"
    Q = G / sub
    assert Q.order == m and Q.is_cyclic(), "G/N is not cyclic of order m"
    assert sub.fast_isomorphic(N), "the subgroup is not isomorphic to N"

count = 0
for N, m in [(Z4, 2), (V4, 2), (Z3, 2), (Z3, 4), (cat(5, "Z5"), 4), (cat(8, "Q8"), 2), (cat(6, "D3"), 3)]:
    for G in cyclic_extensions(N, m):
        check_extension(G, N, m)
        count += 1
print(f"Verified {count} extensions.")

Verified 19 extensions.


## 4. `groups_of_order`

For each prime $p \mid n$ and each group $N$ of order $n/p$ (found recursively), take all cyclic extensions of $N$ by $\mathbb{Z}_p$, and keep one group per isomorphism class. Primes need no special case: the trivial group extended by $\mathbb{Z}_p$ gives $\mathbb{Z}_p$. The groups are relabeled `'0'`, ..., `'n-1'` (identity `'0'`) so that element names don't pile up through the recursion.

In [11]:
def relabel(G, name=None):
    """A copy of G whose elements are named '0', '1', ..., with the identity as '0'."""
    T, e = int_view(G)
    n = len(T)
    new_of = np.empty(n, dtype=np.int64)                 # old index -> new index
    order = [e] + [i for i in range(n) if i != e]
    new_of[order] = np.arange(n)
    table = new_of[T[np.ix_(order, order)]]
    return make_finite_algebra(name or G.name, G.description, [str(i) for i in range(n)], table.tolist())


def groups_of_order(n, cache=None):
    """All SOLVABLE groups of order n, up to isomorphism, built recursively as cyclic extensions by Z_p
    of the groups of order n/p, for each prime p dividing n. Every group of order < 60 is solvable."""
    cache = {} if cache is None else cache
    if n in cache:
        return cache[n]
    if n == 1:
        result = [make_finite_algebra("Z1", "Trivial group", ["0"], [[0]])]
    else:
        result, buckets = [], defaultdict(list)
        for p in sorted(factorint(n)):
            for N in groups_of_order(n // p, cache):
                for G in cyclic_extensions(N, p):
                    key = cheap_invariant(*int_view(G))
                    if any(G.fast_isomorphic(H) for H in buckets[key]):
                        continue
                    buckets[key].append(G)
                    result.append(G)
        result = [relabel(G, name=f"G{n}_{k}") for k, G in enumerate(result, start=1)]
    cache[n] = result
    return result

### Check against OEIS A000001 and finalg's catalog (orders 1 to 40)

For each order, the number of groups found must equal the known count, **and** the groups found must be pairwise non-isomorphic and map onto exactly the groups in `small_groups(n)`.

In [12]:
expected = (1, 1, 1, 2, 1, 2, 1, 5, 2, 2, 1, 5, 1, 2, 1, 14, 1, 5, 1, 5,
            2, 2, 1, 15, 2, 2, 5, 4, 1, 4, 1, 51, 1, 2, 1, 14, 1, 2, 2, 14)    # OEIS A000001

cache, all_ok, found32 = {}, True, set()
print(f"{'n':>3} {'found':>6} {'A000001':>8} {'matches catalog':>18} {'seconds':>8}")
for n in range(1, 41):
    t0 = time.perf_counter()
    gs = groups_of_order(n, cache)
    dt = time.perf_counter() - t0
    names = [identify_small_group(g).name for g in gs]
    ok = (len(gs) == expected[n - 1]
          and len(set(names)) == len(names)
          and set(names) == {g.name for g in small_groups(n)})
    all_ok &= ok
    if n == 32:
        found32 = set(names)
    print(f"{n:>3} {len(gs):>6} {expected[n - 1]:>8} {'yes' if ok else 'NO':>18} {dt:>8.1f}")
assert all_ok
print("\nAll orders 1-40 agree.")

  n  found  A000001    matches catalog  seconds
  1      1        1                yes      0.0
  2      1        1                yes      0.0
  3      1        1                yes      0.0
  4      2        2                yes      0.0
  5      1        1                yes      0.0
  6      2        2                yes      0.0
  7      1        1                yes      0.0
  8      5        5                yes      0.0
  9      2        2                yes      0.0
 10      2        2                yes      0.0
 11      1        1                yes      0.0
 12      5        5                yes      0.0
 13      1        1                yes      0.0
 14      2        2                yes      0.0
 15      1        1                yes      0.0
 16     14       14                yes      0.1
 17      1        1                yes      0.0
 18      5        5                yes      0.0
 19      1        1                yes      0.0
 20      5        5                yes  

Among the 51 groups of order 32 are the three that, in the catalog, had to be imported from GAP because none of direct, semidirect or central products reach them. Cyclic extensions find them too:

In [13]:
print(sorted({"SG32_8", "SG32_15", "SG32_32"} & found32))

['SG32_15', 'SG32_32', 'SG32_8']


### Beyond order 40

The same code gets every order up to 59 right. At order 60 it finds 12 of the 13 groups, because $A_5$ is not solvable and so is not a cyclic extension of anything.

In [14]:
for n, known in [(48, 52), (54, 15), (56, 13), (60, 13)]:
    t0 = time.perf_counter()
    found = len(groups_of_order(n, cache))
    print(f"n = {n}: found {found}, A000001 says {known}   ({time.perf_counter() - t0:.1f} s)")

n = 48: found 52, A000001 says 52   (5.8 s)
n = 54: found 15, A000001 says 15   (1.8 s)
n = 56: found 13, A000001 says 13   (0.8 s)
n = 60: found 12, A000001 says 13   (1.2 s)


## Notes and next steps

* **Where `fast_isomorphic` is used.** Both the within-extension and the cross-extension deduplication use `fast_isomorphic`, after bucketing the candidates by a cheap invariant (for each element: its order, the size of its centralizer, the order of its square, and its number of square roots). Nothing here calls `isomorphic`.
* **Limit of `automorphisms`.** It tries every assignment of candidate images to the generators, which is fine up to $|\mathrm{Aut}|$ in the tens of thousands (it handled $\mathrm{GL}(4,2)$ in a fraction of a second) but not for $\mathbb{Z}_2^5$, where $|\mathrm{Aut}| = 9{,}999{,}360$. The fix is to assign generators one at a time and prune on partial consistency (or to use orbit-stabilizer on the generators). That would be needed for order 64 and up.
* **Nonsolvable groups** (first one: $A_5$, order 60) need a different route: permutation groups, `generate_alternating_group`, or GAP imports. A natural extension is to build the nonsolvable groups of small order from simple groups by direct and semidirect products.
* **Where the time goes.** Order 32 takes on the order of 15 seconds, almost all of it inside `fast_isomorphic`, spent on pairs of non-isomorphic groups that the bucketing invariant does not separate (computing $\mathrm{Aut}(\mathbb{Z}_2^4)$ itself takes about 0.3 s). Strengthening that invariant is the cheap lever (adding the order of the square and the number of square roots roughly halved the time), and a canonical form for tables would remove the pairwise tests altogether.
* **Possible integration into finalg.** `automorphisms`, `inner_automorphisms` and `automorphism_group` fit naturally as `Group` methods, and `cyclic_extensions(N, m)` as `Group.cyclic_extensions(m)` next to `semidirect_product` and `central_product`. `groups_of_order` could live in a new module such as `finalg/group_construction.py`.